<a href="https://colab.research.google.com/github/likitreddy/gen-ai-content-transformation-platform/blob/main/Copy_of_AI_content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio huggingface_hub pypdf python-docx

import gradio as gr
from huggingface_hub import InferenceClient
from pypdf import PdfReader
import docx
import io
import traceback

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
MODEL = "meta-llama/Llama-3.1-8B-Instruct"

client = InferenceClient(model=MODEL, token=HF_TOKEN)

In [ ]:
def extract_text(file):
    if file is None:
        return ""
    name = file.name.lower()
    try:
        if name.endswith(".pdf"):
            reader = PdfReader(file.name)
            return "\n".join(page.extract_text() or "" for page in reader.pages)
        elif name.endswith(".docx"):
            d = docx.Document(file.name)
            return "\n".join(p.text for p in d.paragraphs)
        elif name.endswith(".txt"):
            with open(file.name, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
        else:
            return "Unsupported file type. Please upload PDF, DOCX, or TXT."
    except Exception as e:
        return f"Error reading file: {e}"

In [ ]:
PROMPTS = {
    "Summary": "Summarize the following content in {detail} (audience: {audience}, tone: {tone}, language: {language}):\n\n{content}",
    "Executive Summary": "Write a concise executive briefing (bullet points) for a {audience} audience, {tone} tone, in {language}, {detail}, from:\n\n{content}",
    "MCQs": "Create 5 multiple-choice questions (4 options each, mark the correct answer clearly) testing understanding of the following content. Audience: {audience}. Language: {language}.\n\n{content}",
    "LinkedIn Post": "Turn the following into a professional LinkedIn post. Audience: {audience}. Tone: {tone}. Language: {language}.\n\n{content}",
    "Twitter/X Post": "Turn the following into a Twitter/X thread (3-5 numbered tweets). Tone: {tone}. Language: {language}.\n\n{content}",
    "Advisory": "Turn the following into a structured advisory document (Title, Summary, Key Points, Recommendations). Audience: {audience}. Language: {language}.\n\n{content}",
    "Infographic": "Extract infographic-ready content: headline, 4-6 key stats/points, and a layout suggestion. Language: {language}.\n\n{content}",
    "Presentation": "Create a 5-slide outline (title + 3 bullets per slide) with speaker notes. Audience: {audience}. Tone: {tone}. Language: {language}.\n\n{content}",
    "Video Script": "Create a video package: script, 4-5 scene descriptions, narration text, subtitles, and 2 visual recommendations. Language: {language}.\n\n{content}",
}

DETAIL_MAP = {
    "Brief": "a brief overview (~100 words)",
    "Standard": "a standard-length summary (~200 words)",
    "Detailed": "a detailed breakdown (~400 words)",
}

In [ ]:
def generate(content, file, selected_outputs, audience, tone, language, detail):
    file_text = extract_text(file)
    full_content = (content or "").strip()
    if file_text:
        full_content = (full_content + "\n\n" + file_text).strip()

    if not full_content:
        return " Please paste text or upload a file.", None
    if not selected_outputs:
        return " Please select at least one output type.", None

    results = []
    for output_type in selected_outputs:
        prompt = PROMPTS[output_type].format(
            content=full_content[:12000],
            audience=audience,
            tone=tone,
            language=language,
            detail=DETAIL_MAP[detail],
        )
        try:
            response = client.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=800,
            )
            text = response.choices[0].message.content
        except Exception as e:
            text = f" Generation failed for {output_type}: {e}"
        results.append(f"## {output_type}\n\n{text}")

    final_output = "\n\n---\n\n".join(results)

    out_path = "/content/generated_output.md"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(final_output)

    return final_output, out_path

In [ ]:
with gr.Blocks(title="Content Transformation Platform") as demo:
    gr.Markdown("# Gen AI Platform for Automated Content Transformation — SIH26154")

    with gr.Row():
        with gr.Column(scale=1):
            content_input = gr.Textbox(
                label="Paste Source Content",
                placeholder="Article, report, advisory, threat intel, prompt, etc.",
                lines=10,
            )
            file_input = gr.File(label="Or Upload a File (PDF / DOCX / TXT)")

            output_types = gr.CheckboxGroup(
                choices=list(PROMPTS.keys()),
                label="Select Output Type(s)",
            )

            with gr.Row():
                audience = gr.Dropdown(
                    ["General Public", "Technical Team", "Executives", "Students", "Policy Makers"],
                    value="General Public", label="Target Audience",
                )
                tone = gr.Dropdown(
                    ["Neutral", "Formal", "Conversational", "Urgent"],
                    value="Neutral", label="Tone",
                )
            with gr.Row():
                language = gr.Dropdown(
                    ["English", "Hindi", "Telugu"], value="English", label="Language",
                )
                detail = gr.Radio(
                    ["Brief", "Standard", "Detailed"], value="Standard", label="Level of Detail",
                )

            generate_btn = gr.Button("Generate", variant="primary")
            clear_btn = gr.Button("Clear")

        with gr.Column(scale=1):
            output_box = gr.Markdown(label="Generated Output")
            download_file = gr.File(label="Download Output")

    generate_btn.click(
        fn=generate,
        inputs=[content_input, file_input, output_types, audience, tone, language, detail],
        outputs=[output_box, download_file],
    )
    clear_btn.click(
        fn=lambda: ("", None, [], "General Public", "Neutral", "English", "Standard", "", None),
        inputs=[],
        outputs=[content_input, file_input, output_types, audience, tone, language, detail, output_box, download_file],
    )

demo.launch(share=True, debug=True)